In [3]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

###########################
# Data Manipulation / Model
###########################


# Connect to database via CRUD Module
db = AnimalShelter()

# Get all records from MongoDB
df = pd.DataFrame.from_records(db.read({}))

if '_id' in df.columns:
    df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()


app.layout = html.Div([
    html.Center([
        html.Img(
            src='data:impage/png;base64,{}'.format(encoded_image),
            style={'height': '120px'}
        ),
        html.H1('CS-340 Grazioso Salvare Dashboard'),
        html.H3('Created by Chris Tagart')
    ]),
        

    html.Hr(),
    
    html.Div([
        html.Label('Select Rescue Type:', style={'fontWeight': 'bold'}),

        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset', 'value': 'Reset'},
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'}
            ],
            value='Reset'
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {'name': i, 'id': i, 'deletable': False, 'selectable': True}
            for i in df.columns
        ],
        data=df.to_dict('records'),

        # Interactive table options
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],

        style_table={
            'overflowX': 'auto'
        },

        style_cell={
            'textAlign': 'left',
            'minWidth': '120px',
            'width': '150px',
            'maxWidth': '180px',
            'whiteSpace': 'normal'
        },

        style_header={
            'fontWeight': 'bold'
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%'}
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%'}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    """Update the data table based on selected rescue type."""

    # Reset returns all records
    if filter_type == 'Reset':
        query = {}

    # Water Rescue
    elif filter_type == 'Water':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$regex': 'Labrador Retriever|Chesapeake Bay Retriever|Newfoundland',
                '$options': 'i'
            },
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }

    # Mountain or Wilderness Rescue
    elif filter_type == 'Mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$regex': 'German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler',
                '$options': 'i'
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }

    # Disaster or Individual Tracking
    elif filter_type == 'Disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$regex': 'Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler',
                '$options': 'i'
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 20,
                '$lte': 300
            }
        }

    else:
        query = {}

    # Read matching records from MongoDB
    filtered_df = pd.DataFrame.from_records(db.read(query))

    # Drop MongoDB ObjectId column if it exists
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    return filtered_df.to_dict('records')


@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    """Update chart based on filtered data table."""

    if viewData is None or len(viewData) == 0:
        return [html.Div('No data available for chart.')]

    dff = pd.DataFrame.from_dict(viewData)

    if 'breed' not in dff.columns:
        return [html.Div('Breed column not found.')]

    breed_counts = dff['breed'].value_counts().nlargest(10).reset_index()
    breed_counts.columns = ['breed', 'count']

    fig = px.bar(
        breed_counts,
        x='breed',
        y='count',
        title='Top 10 Breeds in Current Filter',
        labels={
            'breed': 'Breed',
            'count': 'Number of Animals'
        }
    )

    fig.update_layout(xaxis_tickangle=-45)

    return [dcc.Graph(figure=fig)]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    """Highlight selected columns in the data table."""

    if selected_columns is None:
        selected_columns = []

    return [
        {
            'if': {'column_id': i},
            'background_color': '#D2F3FF'
        }
        for i in selected_columns
    ]


@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, index):
    """Update map based on selected row in the data table."""

    if viewData is None or len(viewData) == 0:
        return [html.Div('No data available for map.')]

    dff = pd.DataFrame.from_dict(viewData)

    # Use first row if nothing is selected
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Prevent row index errors
    if row >= len(dff):
        row = 0

    # Check that required map columns exist
    if 'location_lat' not in dff.columns or 'location_long' not in dff.columns:
        return [html.Div('Location columns not found.')]

    lat = dff.loc[row, 'location_lat']
    lon = dff.loc[row, 'location_long']

    # Austin, TX as backup location
    if pd.isna(lat) or pd.isna(lon):
        lat = 30.75
        lon = -97.48

    animal_name = dff.loc[row, 'name'] if 'name' in dff.columns else 'Unknown'
    animal_breed = dff.loc[row, 'breed'] if 'breed' in dff.columns else 'Unknown'

    return [
        dl.Map(
            style={
                'width': '1000px',
                'height': '500px'
            },
            center=[float(lat), float(lon)],
            zoom=10,
            children=[
                dl.TileLayer(id='base-layer-id'),
                dl.Marker(
                    position=[float(lat), float(lon)],
                    children=[
                        dl.Tooltip(animal_breed),
                        dl.Popup([
                            html.H1('Animal Name'),
                            html.P(animal_name),
                            html.H2('Breed'),
                            html.P(animal_breed)
                        ])
                    ]
                )
            ]
        )
    ]


# Run app
app.run_server()

Dash app running on https://donorpromise-userveteran-3000.codio.io/proxy/8050/
